# ScyPibanya — Erde-Mond-Simulation + Gun Club

Gruppe: Nico Hees, Ahmad Alkhaddour, Bastian Klumpp, Youssef Fahmy, Jan Schäfer<br>
DHBW Stuttgart, Scientific Programming Lab

In [ ]:
#imports

import sys
sys.path.insert(0, "src")
sys.path.insert(1, "tests")

from panel import run, render, build_panel
from analysis import sweep_fast, plot_corridor_heatmap

import ipywidgets as widgets
from IPython.display import HTML
import numpy as np
import unittest

## 1. Mathemathische und physikalische Grundlagen

## Definition 1.1 — Newtons Gravitationsgesetz

Zwei Punktmassen $m_1$ und $m_2$ im Abstand $r$ ziehen sich mit der Kraft

$$F = G \cdot \frac{m_1 \cdot m_2}{r^2}$$

an, wobei

$$G = 6.67430 \times 10^{-11}\ \text{m}^3\text{kg}^{-1}\text{s}^{-2}$$

die Gravitationskonstante ist. Die Kraft wirkt entlang der Verbindungslinie beider Körper und ist stets anziehend.

---

## Satz 1.2 — Bewegungsgleichung (Mehrkörperproblem)

Aus Newtons zweitem Axiom $F = m \cdot a$ und Definition 1.1 folgt für die Gesamtbeschleunigung eines Körpers $i$ unter dem Einfluss aller anderen Körper $j$:

$$\vec{a}_i = \sum_{j \neq i} G \cdot \frac{m_j}{r_{ij}^2} \cdot \hat{r}_{ij}$$

Für mehr als zwei Körper besitzt dieses gekoppelte Differentialgleichungssystem keine geschlossene analytische Lösung (**Mehrkörperproblem**). Wir lösen es deshalb numerisch (siehe Definition 1.3).

> **Numerische Absicherung:** Um eine Division durch Null bei extrem nahen Körpern zu vermeiden, wird die paarweise Kraftberechnung nur für $r > 10^{-12}\,\text{m}$ ausgewertet — ohne physikalische Bedeutung, rein numerisch.

---

## Definition 1.3 — Numerische Zeitintegration

Da die Bewegungsgleichung aus Satz 1.2 nicht analytisch lösbar ist, nähern wir uns der Bahn in diskreten Zeitschritten $\Delta t$. Wir implementieren zwei Verfahren mit unterschiedlicher Genauigkeit an:

**Explizites Euler-Verfahren**

Geschwindigkeit und Position werden direkt aus dem alten Zustand fortgeschrieben:

$$\vec{v}_{n+1} = \vec{v}_n + \vec{a}_n \cdot \Delta t \qquad \vec{x}_{n+1} = \vec{x}_n + \vec{v}_n \cdot \Delta t$$

Einfach und schnell, akkumuliert aber pro Schritt einen Fehler erster Ordnung — bei langen Simulationszeiten (z. B. mehrere Mondumläufe) wächst dieser Fehler spürbar an.

**Störmer-Verlet-Verfahren**

Statt der Geschwindigkeit wird die vorherige Position $\vec{x}_{n-1}$ gespeichert, und die neue Position direkt aus den letzten beiden Positionen plus Beschleunigung berechnet:

$$\vec{x}_{n+1} = 2\vec{x}_n - \vec{x}_{n-1} + \vec{a}_n \cdot \Delta t^2$$

Die Geschwindigkeit wird nachträglich per zentralem Differenzenquotient geschätzt:

$$\vec{v}_n = \frac{\vec{x}_{n+1} - \vec{x}_{n-1}}{2\Delta t}$$

Da beim allerersten Schritt noch keine Vorgänger-Position existiert, wird sie einmalig per Euler-Rückwärtsschritt initialisiert ($\vec{x}_{-1} = \vec{x}_0 - \vec{v}_0 \cdot \Delta t$).

---

## Definition 1.4 — Fluchtgeschwindigkeit

Die Fluchtgeschwindigkeit $v_{esc}$ ist die minimale Startgeschwindigkeit, mit der ein Körper das Gravitationsfeld einer Masse $M$ mit Radius $R$ vollständig verlassen kann:

$$v_{esc} = \sqrt{\frac{2GM}{R}}$$

Für die Erde ergibt sich mit `EARTH_MASS` und `EARTH_RADIUS`:

$$v_{esc} \approx 11{,}19\ \text{km/s} \quad (\texttt{constants.EARTH\_ESCAPE\_VELOCITY})$$

Dieser Wert markiert die harte untere Geschwindigkeitsgrenze unseres Trefferkorridors (Abschnitt 1.7): unterhalb dieser Schwelle fällt jedes Geschoss unabhängig vom Abschusswinkel zur Erde zurück.

---

## Definition 1.5 — Kreisbahngeschwindigkeit

Damit ein Körper im Abstand $r$ um eine Zentralmasse $M$ auf stabiler Kreisbahn bleibt, muss die Gravitationskraft genau der Zentripetalkraft entsprechen:

$$G\frac{Mm}{r^2} = \frac{mv^2}{r} \quad \Rightarrow \quad v_{circ} = \sqrt{\frac{GM}{r}}$$

Mit `EARTH_MASS` und `EARTH_MOON_DISTANCE` ergibt sich:

$$v_{circ} \approx 1018\ \text{m/s} \quad (\texttt{constants.MOON\_CIRCULAR\_VELOCITY})$$

— genau die Startgeschwindigkeit, die der Mond in unserem Erde-Mond-Szenario in +Y-Richtung erhält, während die Erde im Ursprung ruht.

---

## Definition 1.6 — Vollständig inelastische Kollision

Berühren sich zwei Körper, gehen wir laut Aufgabenstellung von einer 100 % inelastischen Kollision aus: Beide Körper verschmelzen zu einem, Masse und Impuls bleiben erhalten:

$$m_{ges} = m_1 + m_2 \qquad \vec{v}_{neu} = \frac{m_1\vec{v}_1 + m_2\vec{v}_2}{m_1+m_2}$$

Ein Treffer des Mondes äußert sich im Simulationsverlauf dadurch, dass ein neuer, verschmolzener Körper auftaucht und einer der ursprünglichen Namen (Mond, Geschoss) verschwindet.

---

## 1.7 Der Trefferkorridor

Ein erster grober Test (Abschnitt 4, drei Geschwindigkeiten × drei Winkel, 1 Tag Simulationsdauer) zeigte: Bei 7 und 9 km/s — beide unterhalb der Fluchtgeschwindigkeit aus Definition 1.4 (≈ 11.19 km/s) — fällt das Geschoss unabhängig vom Winkel zur Erde zurück. Bei 12 km/s reicht die Geschwindigkeit zwar aus, um die Erde zu verlassen, aber keiner der drei groben Winkel (0°, 15°, 30°) trifft den Mond.

Das deutet bereits an, dass ein Treffer nicht an einem einzelnen Parameterpaar hängt, sondern nur in einem schmalen Bereich möglich ist. Um diesen Bereich sichtbar zu machen, haben wir mit `sweep_fast()` ein deutlich feineres Raster ausgewertet:

- **51 Geschwindigkeiten:** 10.5–13.0 km/s, Schritt 50 m/s
- **41 Winkel:** 2°–22°, Schritt 0.5°
- **≈ 2000 Zellen**, ausgewertet in wenigen Sekunden dank vektorisierter Verlet-Propagation durch eine einmal vorberechnete Erde-Mond-Bahn

Das Ergebnis (`plot_corridor_heatmap`) bestätigt die Vermutung: Es gibt keinen isolierten Trefferpunkt, sondern einen zusammenhängenden **Trefferkorridor** — ein schmales, diagonal verlaufendes Band im Geschwindigkeit-Winkel-Raum.

Je höher die Startgeschwindigkeit, desto kleiner der nötige Vorhaltewinkel (von rund 21° bei 11.3 km/s bis auf rund 8–9° bei 13 km/s). Das ist physikalisch plausibel — ein schnelleres Geschoss braucht weniger Flugzeit, wodurch der Mond auf seiner Umlaufbahn weniger Zeit hat, sich weiterzubewegen, und ein kleinerer Vorhalt genügt.

Nach links ist der Korridor durch eine harte Wand begrenzt: Die weiße gestrichelte Linie bei $v_{esc} \approx 11.19$ km/s markiert exakt die Fluchtgeschwindigkeit aus Definition 1.4. Links davon bleibt die Farbe der Heatmap durchgehend dunkel (= große Distanz zum Mond) — kein einziger Treffer, egal bei welchem Winkel.

Das erklärt auch rückblickend, warum die groben Testwerte (7, 9, 12 km/s × 0°, 15°, 30°) keinen Treffer erzielten: 7 und 9 km/s lagen unter der Wand, und bei 12 km/s waren alle drei getesteten Winkel zu weit vom schmalen Korridor entfernt, der bei dieser Geschwindigkeit nur etwa 11–12° breit ist.



## 2. Teil 1: Erde-Mond-System

### Aufbau der Simulation

Der Erdmittelpunkt liegt im Koordinatenursprung $(0,0,0)$, wie in der Aufgabenstellung
vorgegeben. Der Mond startet auf der positiven X-Achse im Abstand `EARTH_MOON_DISTANCE`
und bewegt sich in +Y-Richtung mit der Kreisbahngeschwindigkeit `MOON_CIRCULAR_VELOCITY`.
Dadurch bleibt seine Umlaufbahn vollständig in der X/Y-Ebene, was die Visualisierung
stark vereinfacht.

Als Zeitschritt verwenden wir `DEFAULT_EARTH_MOON_TIME_STEP = 1 h`. Ein realer
Mondumlauf dauert ca. 27 Tage — ein deutlich feinerer Zeitschritt würde die
Simulation unnötig verlangsamen, ohne die Genauigkeit spürbar zu verbessern.

**Recherchierte Referenzwerte:**

| Größe | Wert | Quelle |
|---|---|---|
| Erdmasse | 5.972 × 10²⁴ kg | NASA Earth Fact Sheet |
| Erdradius | 6.371 × 10⁶ m | NASA Earth Fact Sheet |
| Mondmasse | 7.346 × 10²² kg | NASA Moon Fact Sheet |
| Mondradius | 1.7374 × 10⁶ m | NASA Moon Fact Sheet |
| Erde-Mond-Abstand | 3.844 × 10⁸ m | NASA Moon Fact Sheet |
| Siderische Umlaufzeit | 27.322 Tage | NASA Moon Fact Sheet |

Die folgende Zelle simuliert 30 Tage — das entspricht etwas mehr als einer vollen
Mondumlaufperiode und eignet sich damit gut zur visuellen Kontrolle:

In [ ]:
emviz = run(False, "Verlet", 30, None, None, 15000)
html = render(emviz)
display(html)

## 3. Teil 2.1: Die Columbiade (Kanonenschuss): Gibt es einen Treffer?

### Schüsse mit verschiedenen Winkeln und Geschwindigkeiten

- 7km/s     0°
- 9km/s     0°
- 12km/s    0°
- 7km/s     15°
- 9km/s     15°
- 12km/s    15°
- 7km/s     30°
- 9km/s     30°
- 12km/s    30°

Es werden alle 9 Simulationen berechnet, eine Geschwindigkeit von 7 oder 9 km/s ist nicht ausreichend, um den Mond zu treffen, da die Gravitation der Erde stärker ist und die Kanonenkugeln zurück auf die Erde fallen. Erst bei 12 km/s ist die Geschwindigkeit stark genug, jedoch trifft keiner der Winkel.

In [ ]:
speeds = [7, 9, 12]
angles = [0, 15, 30]

for speed in speeds:
    for angle in angles:
        display(HTML(f"<h3>Geschwindigkeit: {speed} km/s | Winkel: {angle}°</h3>"))
        try:
            viz = run(True, "Verlet", 1, angle, speed * 1000, 15000)
            html = render(viz)
            display(html)
        except Exception as e:
            print(f"Fehler in Winkel {angle} und Geschwindigkeit {speed}: {e}")

### Simulation mit variablen Werten

In [ ]:
panel, refs = build_panel()
display(panel)

### Wann wird der Mond getroffen?

In [ ]:
analyse_speeds = np.arange(10.5, 13.01, 0.05) * 1000   # 51 speeds
analyse_winkel = np.arange(2.0, 22.01, 0.5)            # 41 angles

korridor = sweep_fast(analyse_speeds, analyse_winkel)   # ~8 s for ~2000 cells
ax = plot_corridor_heatmap(korridor)

## 4. Tests

In [ ]:
# execute all tests
loader = unittest.TestLoader()
suite = loader.discover(start_dir="tests", pattern="test_*.py")
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 5. Zusammenfassung und Quellen

- NASA Earth Fact Sheet: https://nssdc.gsfc.nasa.gov/planetary/factsheet/earthfact.html
- NASA Moon Fact Sheet: https://nssdc.gsfc.nasa.gov/planetary/factsheet/moonfact.html
- Leifiphysik Gravitation: https://www.leifiphysik.de/mechanik/gravitationsgesetz-und-feld/grundwissen/gravitationsgesetz-von-newton
- Gravitation: https://en.wikipedia.org/wiki/Newton's_law_of_universal_gravitation
- Verlet: https://de.wikipedia.org/wiki/Verlet-Algorithmus
- Verlet: https://www.algorithm-archive.org/contents/verlet_integration/verlet_integration.html
- Inelastische Kollision: https://www.leifiphysik.de/mechanik/impulserhaltung-und-stoesse/grundwissen/zentraler-vollkommen-unelastischer-stoss